# Outlier — Week 1: Data Validation

**Goal:** confirm the data foundation before any modeling — that our scene segmentation lines up with TRIPOD's gold turning-point labels, that the labels load, and that the evaluation metric behaves.

This notebook uses the `outlier` package in `src/`. Path B stack: sentence-transformers (later) instead of USE; **no old TRIPOD/USE environment required.**

In [5]:
import sys
from pathlib import Path
# locate the repo root (the folder containing src/outlier) whether run from repo root or notebooks/
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'src' / 'outlier').exists()), here)
sys.path.insert(0, str(root / 'src'))

try:
    import pandas as pd
except ModuleNotFoundError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
from outlier import data, metrics, config
print('repo root:', root)
print('turning points:', config.TP_NAMES)

repo root: c:\Users\porte\OneDrive - Neumont College of Computer Science\Documents\.AllRelevantProjects\aml-script-analyzer
turning points: ['Opportunity', 'Change of Plans', 'Point of No Return', 'Major Setback', 'Climax']


## 1. Gold labels & segmentation alignment

The 15 test movies have **scene-level** gold turning points. Our segmenter must produce scene indices that match — i.e. every gold TP index must be < the movie's scene count.

In [6]:
gold = data.load_gold_labels()
rows = []
for mv, tps in gold.items():
    scenes = data.get_scenes(mv)
    max_idx = max(i for tp in tps for i in tp)
    rows.append({
        'movie': mv,
        'scenes': len(scenes),
        'tp_scenes': [tp[0] for tp in tps],
        'max_gold_idx': max_idx,
        'aligned': max_idx < len(scenes),
    })
df = pd.DataFrame(rows)
print('all aligned:', bool(df['aligned'].all()))
df

all aligned: True


,movie,scenes,tp_scenes,max_gold_idx,aligned
0,The Back-up Plan,132,"[9, 40, 82, 106, 131]",131,True
1,The Shining (film),230,"[6, 38, 76, 177, 223]",229,True
2,Juno (film),116,"[3, 31, 39, 86, 107]",107,True
3,Soldier (1998 American film),230,"[35, 51, 109, 210, 223]",223,True
4,Panic Room,167,"[17, 56, 135, 148, 159]",160,True
5,Arbitrage (film),110,"[35, 57, 67, 105, 109]",109,True
6,The Breakfast Club,42,"[5, 20, 31, 31, 34]",39,True
7,Slumdog Millionaire,196,"[32, 108, 139, 150, 188]",191,True
8,Total Recall (1990 film),163,"[16, 51, 72, 112, 145]",154,True
9,Unforgiven,106,"[6, 53, 88, 95, 98]",98,True


## 2. Turning points on a real film

Sanity check: do the labelled scenes actually look like the beats they claim to be? Here are Die Hard's five turning points (first line of each scene).

In [7]:
mv = 'Die Hard'
scenes = data.get_scenes(mv)
for name, tp in zip(config.TP_NAMES, gold[mv]):
    idx = tp[0]
    first = scenes[idx].strip().splitlines()[0][:70]
    print(f'{name:20} scene {idx:>3}: {first}')

Opportunity          scene  11: 21      INT. ELLIS' OFFICE - NIGHT                             21 
Change of Plans      scene  26: 52      INT. 30th FLOOR (HOSTAGE FLOOR) - SAME                 52
Point of No Return   scene  99: 227     INT. 33RD FLOOR - ON MCCLANE                         227
Major Setback        scene 114: 346     INT. VAULT ROOM                                       346
Climax               scene 116: 393     EXT. BUILDING - LONG SHOT                             393


## 3. Silver training labels

Projected (distant-supervision) TP scene indices for the training movies, from SUMMER. Same shape as gold: `{movie: [tp1..tp5]}`. (If this cell errors with *truncated*, the Git-LFS file is still syncing — re-run once it's fully downloaded.)

In [8]:
try:
    silver = data.load_silver_labels()
    on_disk = set(data.list_movies())
    have_script = sum(1 for m in silver if m in on_disk)
    print(f'silver movies: {len(silver)} | with a screenplay on disk: {have_script}')
    ex = next(iter(silver))
    print('example:', ex, '->', silver[ex])
except Exception as e:
    print('silver not loadable yet (LFS still syncing?):', e)

silver movies: 84 | with a screenplay on disk: 81
example: 10 Things I Hate About You -> [[20, 21, 22], [29, 30, 31], [43, 44, 45], [66, 67, 68], [84, 85, 86]]


## 4. Metric sanity check

`TA` = exact agreement, `PA` = within a small window, `D` = normalized distance (lower is better). A perfect prediction should score TA=1.0 / D=0.0; a random one should not.

In [9]:
g = gold['Die Hard']
n = len(data.get_scenes('Die Hard'))
perfect = [tp[0] for tp in g]
print('perfect :', metrics.evaluate(perfect, g, n))
print('all-zero:', metrics.evaluate([0, 0, 0, 0, 0], g, n))

perfect : {'TA': 1.0, 'PA': 1.0, 'D': 0.0}
all-zero: {'TA': 0.0, 'PA': 0.0, 'D': 0.6203389830508474}


## 5. EDA — where the turning points actually land

Position of each TP as a fraction of total scenes across all gold + clean-silver movies.
The Climax (TP5) is very tight against the end; TP2 and TP3 vary widely — that variance is where the trained model has to actually read the text to be useful.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

silver = data.load_silver_labels(collapse=True)

def collect_positions(label_dict):
    positions = {i: [] for i in range(1, 6)}
    for m, tps in label_dict.items():
        scenes = data.get_scenes(m)
        n = len(scenes)
        if n == 0:
            continue
        # skip out-of-range labels (a few silver movies have those)
        if max(max(tp) for tp in tps) >= n:
            continue
        for i, tp in enumerate(tps, 1):
            mid = tp[len(tp) // 2]
            positions[i].append(mid / n)
    return positions

positions_gold = collect_positions(gold)
positions_silver = collect_positions(silver)

fig, ax = plt.subplots(figsize=(9, 4))
tp_names = ["Opportunity", "Change of Plans", "Point of No Return", "Major Setback", "Climax"]
box_data = [positions_gold[i] + positions_silver[i] for i in range(1, 6)]
bp = ax.boxplot(box_data, labels=[f"TP{i}\n{n}" for i, n in enumerate(tp_names, 1)], showmeans=True)
ax.set_ylabel("Position (fraction of script)")
ax.set_title("Turning-point positions across gold + clean silver")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean positions (gold only):")
for i, name in enumerate(tp_names, 1):
    ps = positions_gold[i]
    print(f"  TP{i} {name:<20} n={len(ps):>2}  mean={np.mean(ps):.2f}  std={np.std(ps):.2f}")


## 6. EDA — scene count distribution

How many scenes a script has directly affects modeling. Very short scripts (like *The Breakfast Club* at 42 scenes) are outliers — the bi-LSTM has less context to work with, and TP positions become quantized (only a few scenes to choose from). Most scripts fall in the 100–200 range.

In [ ]:
counts_gold = [len(data.get_scenes(m)) for m in gold.keys()]
counts_silver = []
for m in silver.keys():
    if m not in data.list_movies():
        continue
    n = len(data.get_scenes(m))
    if n == 0:
        continue
    counts_silver.append(n)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(counts_silver, bins=25, alpha=0.55, label=f"silver (n={len(counts_silver)})", color="#3E7050")
ax.hist(counts_gold, bins=15, alpha=0.75, label=f"gold (n={len(counts_gold)})", color="#C9A961")
ax.set_xlabel("Scenes per script")
ax.set_ylabel("Number of scripts")
ax.set_title("Scene count distribution")
ax.axvline(np.median(counts_silver + counts_gold), color="black", linestyle="--", label=f"median = {int(np.median(counts_silver + counts_gold))}")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"min: {min(counts_silver + counts_gold)}")
print(f"max: {max(counts_silver + counts_gold)}")
print(f"mean: {np.mean(counts_silver + counts_gold):.0f}")
print(f"median: {np.median(counts_silver + counts_gold):.0f}")


## 7. EDA — multi-annotator agreement on silver

SUMMER's silver labels sometimes have three annotator variants per movie (suffixes `_0`, `_1`, `_2`).
For those movies I can measure how consistent the annotators are on where each TP lands.
Small standard deviations = high agreement; large ones = the annotators disagreed.

In [ ]:
import re
raw_silver = data.load_silver_labels(collapse=False)

# Group by movie
by_movie = {}
for k, tps in raw_silver.items():
    movie = re.sub(r"_\d+$", "", k)
    by_movie.setdefault(movie, []).append(tps)

# Keep only movies with 3 annotators and a valid script
multi = {m: v for m, v in by_movie.items()
         if len(v) == 3 and m in data.list_movies() and len(data.get_scenes(m)) > 0}
print(f"movies with 3 annotators + script:  {len(multi)}")

# For each movie, compute per-TP position std across annotators (as fraction of script)
disagreement = np.zeros((len(multi), 5))
labels = list(multi.keys())
for i, m in enumerate(labels):
    n = len(data.get_scenes(m))
    for k in range(5):
        # take the middle of each annotator's acceptable window
        positions = [ann[k][len(ann[k]) // 2] / n for ann in multi[m] if len(ann[k]) > 0]
        disagreement[i, k] = np.std(positions) if len(positions) > 1 else 0.0

fig, ax = plt.subplots(figsize=(7, max(4, len(labels) * 0.25)))
im = ax.imshow(disagreement, aspect="auto", cmap="YlOrRd", vmin=0, vmax=disagreement.max())
ax.set_xticks(range(5))
ax.set_xticklabels(["TP1", "TP2", "TP3", "TP4", "TP5"])
ax.set_yticks(range(len(labels)))
ax.set_yticklabels([m[:30] for m in labels], fontsize=8)
ax.set_title("Multi-annotator disagreement (std of TP position across 3 annotators)")
plt.colorbar(im, ax=ax, label="std (fraction of script)")
plt.tight_layout()
plt.show()

# Aggregate: which TPs are hardest?
print("\nMean per-TP disagreement across the multi-annotator subset:")
for k, name in enumerate(["Opportunity", "Change of Plans", "Point of No Return", "Major Setback", "Climax"]):
    print(f"  TP{k+1} {name:<20}  mean-std = {disagreement[:, k].mean():.3f}")


## 8. Overfit-a-single-batch test

Before training the real model, we prove the pipeline (data → MiniLM → bi-LSTM → loss → backprop) is wired correctly.
Take a tiny fixed set of 3 gold movies, encode them with MiniLM, and train the bi-LSTM head for a few hundred epochs.
If the wiring is correct, the training loss should drop to near zero on this fixed set — the model has enough capacity to memorize 3 scripts, so anything else means the pipeline is broken.

This is the standard sanity check before running full training; it doesn't tell us anything about generalization,
only that the plumbing works.

Runs on CPU in ~1–2 minutes for 3 movies. On GPU it's instant.

In [ ]:
from outlier.embeddings import SceneEncoder
from outlier.model import TPFinder, overfit_batch
import torch

# Pick 3 gold movies with reasonable scene counts (skip Breakfast Club — too short)
overfit_movies = ["Die Hard", "Panic Room", "The Crying Game"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

encoder = SceneEncoder(device=device)
print(f"scene encoder loaded  (dim = {encoder.dim})")

samples = []
for m in overfit_movies:
    scenes = data.get_scenes(m)
    embs = encoder.encode_scenes(scenes)     # (n_scenes, 384) numpy
    samples.append((embs, gold[m]))
    print(f"  {m:<20} scenes={len(scenes):>4}  embs.shape={embs.shape}")

model = TPFinder(embed_dim=encoder.dim)
print(f"model params: {sum(p.numel() for p in model.parameters()):,}")

torch.manual_seed(42)
losses = overfit_batch(model, samples, epochs=300, lr=1e-3, device=device, verbose=True)

# Plot the training curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(losses, color="#1B2A3E", linewidth=1.5)
ax.axhline(0.5, ls="--", color="gray", alpha=0.5, label="informative threshold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean training loss (per script)")
ax.set_title(f"Overfit-a-single-batch — bi-LSTM on {len(samples)} movies")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nfinal loss: {losses[-1]:.4f}   (started at {losses[0]:.4f})")
print("If final loss <~ 0.5, the pipeline works and the model has capacity to memorize 3 scripts.")


## 9. Overfit sanity check — predictions

After overfitting, the model should predict TPs that land inside the acceptable gold sets on the training movies.
This isn't a real evaluation (we trained on these scripts), but it confirms the loss-to-prediction path works.

In [ ]:
from outlier.model import predict_tps

for embs, tps in samples:
    n = embs.shape[0]
    preds = predict_tps(model, embs, device=device)
    print("gold acceptable sets:", tps)
    print("model predictions:   ", preds)
    hits = [1 if p in g else 0 for p, g in zip(preds, tps)]
    print(f"exact hits (TA on train): {sum(hits)}/5    n_scenes={n}")
    print()


## Next — Week 3

1. **Emotional branch** — DistilRoBERTa on scene halves; per-scene valence numbers; emotional-arc curve for Die Hard.
2. **McKee-conformance signals** — flat-scene rate, escalation slope across act climaxes, alternation compliance on the last two act climaxes.
3. **First-pass McKee readout demo** for Die Hard, plotted alongside the (untrained-yet) TP predictions from this overfit run.

Week 4 will train the bi-LSTM on the full 72-silver corpus with hyperparameter tuning + MLflow tracking, and evaluate against the 15 gold movies with TA / PA / D metrics.